# M5 Agentic AI — Customer Service Agent (Planning with Code Execution)


## 1. Introduction

Planning with code execution means letting the LLM write Python that becomes the plan itself.
This notebook simulates a sunglasses store with inventory + transactions, then uses the model
to generate **plan-as-code** that can be executed safely in a sandbox.


## 2. Setup


In [1]:
from __future__ import annotations

import io
import json
import re
import sys
import traceback
from typing import Any, Dict, Optional

from dotenv import load_dotenv
from openai import OpenAI
from tinydb import Query, where

import utils      # helper functions for prompting/printing
import inv_utils  # inventory/transactions/schema helpers

load_dotenv()
client = OpenAI()


### 2.1 Create Example Tables


In [2]:
db, inventory_tbl, transactions_tbl = inv_utils.seed_db()


In [3]:
utils.print_html(json.dumps(inventory_tbl.all(), indent=2), title="Inventory Table")
utils.print_html(json.dumps(transactions_tbl.all(), indent=2), title="Transactions Table")


## 2.2 Prompt


In [4]:
PROMPT = "You are a senior data assistant. PLAN BY WRITING PYTHON CODE USING TINYDB.\n\nDatabase Schema & Samples (read-only):\n{schema_block}\n\nExecution Environment (already imported/provided):\n- Variables: db, inventory_tbl, transactions_tbl  # TinyDB Table objects\n- Helpers: get_current_balance(tbl) -> float, next_transaction_id(tbl, prefix=\"TXN\") -> str\n- Natural language: user_request: str  # the original user message\n\nPLANNING RULES (critical):\n- Derive ALL filters/parameters from user_request. Do NOT hard-code values.\n- Build TinyDB queries dynamically with Query(). If a constraint isn't in user_request, don't apply it.\n- Be conservative: if intent is ambiguous, do read-only (DRY RUN).\n\nTRANSACTION POLICY (hard):\n- Do NOT create aggregated multi-item transactions.\n- If the request contains multiple items, create a separate transaction row PER ITEM.\n- For each item:\n  - compute its own line total (unit_price * qty),\n  - insert ONE transaction with that amount,\n  - update balance sequentially (balance += line_total),\n  - update the item\u2019s stock.\n- If any requested item lacks sufficient stock, do NOT mutate anything; reply with STATUS=\"insufficient_stock\".\n\nHUMAN RESPONSE REQUIREMENT (hard):\n- You MUST set `answer_text` (str) with a short, customer-friendly sentence (1\u20132 lines).\n- This sentence is the only user-facing message.\n\nACTION POLICY:\n- If the request clearly asks to change state (buy/purchase/return/restock/adjust):\n    ACTION=\"mutate\"; SHOULD_MUTATE=True\n  Otherwise:\n    ACTION=\"read\"; SHOULD_MUTATE=False (DRY RUN)\n\nFAILURE & EDGE-CASE HANDLING:\n- Always set STATUS in:\n  \"success\", \"no_match\", \"insufficient_stock\", \"invalid_request\", \"unsupported_intent\".\n- Print a brief log to stdout, e.g., \"LOG: ACTION=read DRY_RUN=True STATUS=no_match\".\n\nOUTPUT CONTRACT:\n- Return ONLY executable Python between:\n  <execute_python> ... </execute_python>\n\nUser request:\n{question}"


## 2.3 Code generation (plan-as-code)


In [5]:
def generate_llm_code(
    prompt: str,
    *,
    inventory_tbl,
    transactions_tbl,
    model: str = "gpt-4.1-mini",
    temperature: float = 0.2,
) -> str:
    schema_block = inv_utils.build_schema_block(inventory_tbl, transactions_tbl)
    prompt = PROMPT.format(schema_block=schema_block, question=prompt)

    resp = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": "You write safe, well-commented TinyDB code to handle data questions and updates."},
            {"role": "user", "content": prompt},
        ],
    )
    return resp.choices[0].message.content or ""


## 2.4 Executor (run a given plan)


In [6]:
def _extract_execute_block(text: str) -> str:
    '''Returns code inside <execute_python>...</execute_python>. If missing, assumes raw Python.'''
    if not text:
        raise RuntimeError("Empty content passed to code executor.")
    m = re.search(r"<execute_python>(.*?)</execute_python>", text, re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else text.strip()


def execute_generated_code(
    code_or_content: str,
    *,
    db,
    inventory_tbl,
    transactions_tbl,
    user_request: Optional[str] = None,
) -> Dict[str, Any]:
    '''Execute code in a controlled namespace; capture stdout/errors and extract answer_text/rows/json.'''
    code = _extract_execute_block(code_or_content)

    SAFE_GLOBALS = {
        "Query": Query,
        "get_current_balance": inv_utils.get_current_balance,
        "next_transaction_id": inv_utils.next_transaction_id,
        "user_request": user_request or "",
    }
    SAFE_LOCALS = {
        "db": db,
        "inventory_tbl": inventory_tbl,
        "transactions_tbl": transactions_tbl,
    }

    _stdout_buf, _old_stdout = io.StringIO(), sys.stdout
    sys.stdout = _stdout_buf
    err_text = None
    try:
        exec(code, SAFE_GLOBALS, SAFE_LOCALS)
    except Exception:
        err_text = traceback.format_exc()
    finally:
        sys.stdout = _old_stdout

    printed = _stdout_buf.getvalue().strip()

    answer = (
        SAFE_LOCALS.get("answer_text")
        or SAFE_LOCALS.get("answer_rows")
        or SAFE_LOCALS.get("answer_json")
    )

    return {
        "code": code,
        "stdout": printed,
        "error": err_text,
        "answer": answer,
        "transactions_tbl": transactions_tbl.all(),
        "inventory_tbl": inventory_tbl.all(),
    }


## 2.5 Demo: read-only query


In [7]:
prompt_round = "Do you have any round sunglasses in stock that are under $100?"
full_content_round = generate_llm_code(
    prompt_round,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    model="o4-mini",
    temperature=1.0,
)
utils.print_html(full_content_round, title="Plan with Code (Full Response)")

result = execute_generated_code(
    full_content_round,
    db=db,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    user_request=prompt_round,
)
utils.print_html(result["answer"], title="Plan Execution · Extracted Answer")


## 2.6 Demo: return mutation


In [8]:
prompt_aviator = "Return 2 Aviator sunglasses I bought last week."
full_content_aviator = generate_llm_code(
    prompt_aviator,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    model="o4-mini",
    temperature=1.0,
)
utils.print_html(full_content_aviator, title="Plan with Code (Full Response)")

result = execute_generated_code(
    full_content_aviator,
    db=db,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    user_request=prompt_aviator,
)
utils.print_html(result["answer"], title="Plan Execution · Extracted Answer")


## 3. Customer service agent wrapper


In [9]:
def customer_service_agent(
    question: str,
    *,
    db,
    inventory_tbl,
    transactions_tbl,
    model: str = "o4-mini",
    temperature: float = 1.0,
    reseed: bool = False,
) -> dict:
    '''Question → plan-as-code → safe execution → before/after snapshots.'''

    if reseed:
        inv_utils.create_inventory()
        inv_utils.create_transactions()

    utils.print_html(question, title="User Question")

    full_content = generate_llm_code(
        question,
        inventory_tbl=inventory_tbl,
        transactions_tbl=transactions_tbl,
        model=model,
        temperature=temperature,
    )
    utils.print_html(full_content, title="Plan with Code (Full Response)")

    utils.print_html(json.dumps(inventory_tbl.all(), indent=2), title="Inventory Table · Before")
    utils.print_html(json.dumps(transactions_tbl.all(), indent=2), title="Transactions Table · Before")

    exec_res = execute_generated_code(
        full_content,
        db=db,
        inventory_tbl=inventory_tbl,
        transactions_tbl=transactions_tbl,
        user_request=question,
    )

    utils.print_html(exec_res["answer"], title="Plan Execution · Extracted Answer")
    utils.print_html(json.dumps(inventory_tbl.all(), indent=2), title="Inventory Table · After")
    utils.print_html(json.dumps(transactions_tbl.all(), indent=2), title="Transactions Table · After")

    return {
        "full_content": full_content,
        "exec": {
            "code": exec_res["code"],
            "stdout": exec_res["stdout"],
            "error": exec_res["error"],
            "answer": exec_res["answer"],
            "inventory_after": inventory_tbl.all(),
            "transactions_after": transactions_tbl.all(),
        },
    }


## 4. Try it out


In [10]:
prompt = "I want to buy 3 pairs of classic sunglasses and 1 pair of aviator sunglasses."
out = customer_service_agent(
    prompt,
    db=db,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    model="o4-mini",
    temperature=1.0,
    reseed=True,
)
